In [1]:
# TODO: documentation...

In [2]:
# Notebook config

# path to Hugging Face tokenizer files
MMFREELM_370M_HF_TOKENIZER_PATH = "/Users/cuongwilliams/.cache/huggingface/hub/models--ridger--MMfreeLM-370M/snapshots/b5b01a2367ea983c8e753614ccda73c5c68d7190//tokenizer.json"
MMFREELM_370M_HF_TOKENIZER_CONFIG_PATH = "/Users/cuongwilliams/.cache/huggingface/hub/models--ridger--MMfreeLM-370M/snapshots/b5b01a2367ea983c8e753614ccda73c5c68d7190/tokenizer_config.json"

# path to my bytesarray exports
MMFREELM_370M_HF_MODEL_TOC = "/Users/cuongwilliams/Projects/scratchnets/scratchnets/data/models/MMfreeLM-370M/hf_export/MMfreeLM-370M_toc.json"
MMFREELM_370M_HF_MODEL_BIN = "/Users/cuongwilliams/Projects/scratchnets/scratchnets/data/models/MMfreeLM-370M/hf_export/MMfreeLM-370M.bin"

In [3]:
# Imports

# Built-in packages
import sys
import pickle
import zipfile
import io
import json
import base64
import importlib
from collections import defaultdict
from collections import UserDict, OrderedDict
from typing import TYPE_CHECKING, Any, Dict, List, NamedTuple, Optional, Sequence, Tuple, Union
import math
import types

# External packages
import numpy as np
from tokenizers import Tokenizer, AddedToken, Encoding
import ml_dtypes

In [4]:
# Dynamic imports - create torch "stubs" dynamically (needed to unpickle python tensors in pth model file )

class BFloat16Storage:
    #np.float16
    dtype = ml_dtypes.bfloat16
    nbytes = 2
def _rebuild_tensor_v2(a,b,c,d,e,f):
    '''Stub for the pickled pytorch call.'''
    #print("rebuild!",a,b,c,d,e,f)
    tensor = a.reshape(c)
    return tensor
_utils = types.ModuleType("_utils")
_utils._rebuild_tensor_v2 = _rebuild_tensor_v2
_utils._rebuild_tensor_v2 = _rebuild_tensor_v2
__init__ = types.ModuleType("__init__")
torch = types.ModuleType("torch")
torch.BFloat16Storage = BFloat16Storage
torch._utils = _utils
torch.__init__ = __init__
sys.modules['torch']=torch
#print(sys.modules)
#print(type(torch),dir(torch))
#print(type(torch._utils), dir(torch._utils))
#from torch import _utils
#import torch._utils

In [186]:
# Initialize globals

NUM_LAYERS = 24
NUM_HEADS = 1

In [6]:
# Create the tokenizer via configs

def create_tokenizer():
    global TOKENIZER
    # create tokenizer from configs 
    f = open(MMFREELM_370M_HF_TOKENIZER_CONFIG_PATH,"r")
    contents = f.read()
    f.close()
    tokenizer_config = json.loads(contents)
    items = tokenizer_config["added_tokens_decoder"].items()
    tokens = []
    for idx, token in items:
        if isinstance(token, dict):
            token = AddedToken(**token)
            tokens.append(token)
    tokenizer = Tokenizer.from_file(MMFREELM_370M_HF_TOKENIZER_PATH)
    tokenizer.no_truncation()
    tokenizer.add_tokens( tokens )
    return tokenizer  

In [7]:
# Tokenizes a prompt

def tokenize_prompt( prompt, tokenizer ):
    encoding = tokenizer.encode_batch(prompt)[0]
    encodings = [ encoding ]  
    def convert_encoding(encoding):
        encodings = [ encoding ]
        encoding_dict = defaultdict(list)
        return_token_type_ids= False 
        return_attention_mask= True 
        return_special_tokens_mask= False 
        return_offsets_mapping= False 
        return_length= False
        for e in encodings:
            encoding_dict["input_ids"].append(e.ids)    
            if return_token_type_ids:
                encoding_dict["token_type_ids"].append(e.type_ids)
            if return_attention_mask:
                encoding_dict["attention_mask"].append(e.attention_mask)
            if return_special_tokens_mask:
                encoding_dict["special_tokens_mask"].append(e.special_tokens_mask)
            if return_offsets_mapping:
                encoding_dict["offset_mapping"].append(e.offsets)
            if return_length:
                encoding_dict["length"].append(len(e.ids))
        return encoding_dict, encodings    
    tokens_and_encodings = [ convert_encoding(encoding) for encoding in encodings ]  
    sanitized_tokens = {}
    for key in tokens_and_encodings[0][0].keys():
        stack = [e for item, _ in tokens_and_encodings for e in item[key]]
        sanitized_tokens[key] = stack
    sanitized_encodings = [e for _, item in tokens_and_encodings for e in item]
    token_ids = np.array( sanitized_tokens['input_ids'], np.uint32 )    
    return token_ids, 1, token_ids.shape[1]

In [8]:
# Loads the HF tensor exported model

def load_hf_export_model():
    #print("reading llama_toc")
    with open(MMFREELM_370M_HF_MODEL_TOC,"r") as f:
        llama_toc = json.loads(f.read())
    #print("llama_toc=", llama_toc)  
    #print("reading llama_bin")
    with open(MMFREELM_370M_HF_MODEL_BIN,"rb") as f:
        llama_b = f.read()
    #print("llama_bin size=", len(llama_b))  
    llama_io = io.BytesIO(llama_b)
    fcounter=0
    hf_model = OrderedDict()
    for item in llama_toc.items():   
        # Get the HF exported model key name
        hf_key = item[0]  
        shape, tot_size = item[1] # get the tensor shape and total size captured during export
        array_buf = llama_io.read(tot_size) # read just the total bytes for the tensor
        # Convert the buffer ato  file stream object
        iobuff = io.BytesIO(array_buf)
        iobuff.seek(0)
        # Prepare class for reading the torch save format (zip archive)
        class CustomLoader:
            def __init__(self,f):
                self.f = f # file pointer
                self.zf = None # pointer to zipfile object
                self.unpickler = None # custom unpickler
            def load(self):
                # process file as zipped compressed
                izf = zipfile.is_zipfile(self.f)
                if not izf: raise Exception("ERROR: This loader only supports recent Pytorch versions")
                self.zf = zipfile.ZipFile(self.f)
                #print("zip names=", self.zf.namelist())
                # unzip the TOC section
                toc_bytes = io.BytesIO( self.zf.read("archive/data.pkl") )                 
                def persistent_load(saved_id):
                    typename = saved_id[0]           
                    if typename == 'storage':
                        storage_type, key, location, numel = saved_id[1:]
                        #print(storage_type, key, location, numel)
                        name = f"archive/data/{key}"
                    else:
                        raise Exception("unknown storage")              
                    dp = io.BytesIO( self.zf.read(name) )
                    bytes = dp.read()
                    tot_bytes = numel * 2 # assume bfloat16         
                    #print("bytes=",len(bytes), key, numel, tot_bytes)
                    arr = np.frombuffer(bytes, 
                                        dtype=ml_dtypes.bfloat16, # assume bfloat16
                                        count=numel)
                    return arr     
                class UnpicklerWrapper(pickle.Unpickler):
                    def find_class(self, mod_name, name):
                        #print("GW _legacy_load unpickler find_class", 
                        #      mod_name, name,
                        #      type(mod_name), type(name))
                        #print("UnpicklerWrapper, HF find_class", mod_name, name)
                        if mod_name=="torch._utils" and name=="_rebuild_tensor_v2":
                            return torch._utils._rebuild_tensor_v2
                        else:
                            return super().find_class(mod_name, name)
                    def __init__(self,f):
                        self.f = f
                        super(UnpicklerWrapper, self).__init__(f)
                # Unpickle the TOC object which creates the unpacked model
                self.unpickler = UnpicklerWrapper(toc_bytes)
                self.unpickler.persistent_load = persistent_load    
                result = self.unpickler.load()
                return result
        # read the tensor associated with the HF key
        loader = CustomLoader(iobuff)
        arr = loader.load()
        #print("arr=", arr.shape, arr.dtype)  
        fcounter += tot_size
        hf_model[hf_key] = arr # replace key's value with array  
    return hf_model

In [149]:
# Converts token_ids to embeddings

def token_ids__2__token_embeddings( token_ids, hf_model ):
    # start with the prompt token ids
    #print("token_ids shape=", #token_ids, 
    #      token_ids.shape, token_ids.dtype)
    # get the embedding matrix
    embed_weights = hf_model[ 'model.embeddings.weight' ]
    #print("embed weights shape=", embed_weights.shape, embed_weights.dtype)
    # lookup embeddings using token_ids as index
    #print("idx=", token_ids.squeeze().shape )
    embedding_activations = embed_weights[ None, token_ids.squeeze(), :]
    #print("embedding activations=", embedding_activations, 
    #      embedding_activations.shape, embedding_activations.dtype)
    return embedding_activations

In [151]:
# Applies layer normalization before attention QKV projection

def apply_rms_norm_pre_attention( hidden_state, hf_model, layer ):
    residual = hidden_state
    hidden_state = hidden_state.astype(np.float32)
    #print("hidden_state_32=", hidden_state, 
    #      hidden_state.dtype) 
    variance_epsilon= 1e-06
    rmsnorm_weights = hf_model[ "model.layers.%d.attn_norm.weight" % layer ]
    #print("apply_rms_norm_pre_attention: weights=", rmsnorm_weights, 
    #      rmsnorm_weights.shape)
    # FROM HF LLama3.1 WE USE MEAN variance = np.mean( np.power(hidden_state,2), 2, keepdims=True )
    variance = np.sum( np.power(hidden_state,2), 2, keepdims=True )
    #print("variance=", variance, 
    #      variance.shape)
    vardiv = np.divide( variance, hidden_state.shape[-1] )
    #print("vardiv=", vardiv, 
    #      vardiv.shape, hidden_state.shape[-1] )
    v_sqrt = np.sqrt( vardiv + variance_epsilon )
    #print("v_sqrt=", v_sqrt, 
    #      v_sqrt.shape)
    v_rsqrt = np.reciprocal( v_sqrt  )
    #print("v_rsqrt=", v_rsqrt, 
    #      v_rsqrt.shape)
    hidden_state = np.multiply( hidden_state, v_rsqrt )
    #print("hidden_state_mult=", hidden_state, 
    #      hidden_state.shape)
    normalized_hidden_state = rmsnorm_weights * hidden_state.astype( ml_dtypes.bfloat16 )
    print("apply_rms_norm_pre_attention: normalized_hidden_state=", normalized_hidden_state, 
          normalized_hidden_state.shape,normalized_hidden_state.dtype)
    return residual, normalized_hidden_state

In [148]:
# Applies I projection only

def layer_norm_fwd_quant_kernel( input_hidden_state, rmsnorm_weights  ):
    variance_epsilon= 1e-08
    #print("apply_i_projection: weights=", rmsnorm_weights, 
    #      rmsnorm_weights.shape)
    # FROM HF LLama3.1 WE USE MEAN variance = np.mean( np.power(hidden_state,2), 2, keepdims=True )
    variance = np.sum( np.power(input_hidden_state,2), 2, keepdims=True )
    #print("variance=", variance, 
    #      variance.shape)
    vardiv = np.divide( variance, input_hidden_state.shape[-1] )
    #print("vardiv=", vardiv, 
    #      vardiv.shape, hidden_state.shape[-1] )
    v_sqrt = np.sqrt( vardiv + variance_epsilon )
    #print("v_sqrt=", v_sqrt, 
    #      v_sqrt.shape)
    v_rsqrt = np.reciprocal( v_sqrt  )
    #print("v_rsqrt=", v_rsqrt, 
    #      v_rsqrt.shape)
    hidden_state = np.multiply( input_hidden_state, v_rsqrt )
    #print("apply_i_projection: after internal norm hidden_state=", hidden_state, 
    #      hidden_state.shape)
    normalized_hidden_state = rmsnorm_weights * hidden_state.astype( ml_dtypes.bfloat16 )
    #print("apply_i_projection: normalized_hidden_state=", normalized_hidden_state, 
    #      normalized_hidden_state.shape,normalized_hidden_state.dtype)
    # quantize
    abs_ = np.abs( normalized_hidden_state )
    #print("apply_i_projection: abs=", abs_, abs_.shape)
    max_ = np.max( abs_, axis=2, keepdims=True )
    #print("apply_i_projection: max_", max_, max_.shape)
    smn_ = np.array([[[1e-5]]])
    smn_ = np.broadcast_to( smn_, ( 1, max_.shape[1], 1 ) )
    #print("apply_i_projection: smn_", smn_)
    max__ = np.maximum( max_, smn_)
    max__ = np.broadcast_to( max__, abs_.shape)
    #print("apply_i_projection: max__", max__)
    scale_ = 127.0 / max__
    #print("apply_i_projection: scale_", scale_)
    qtz_ = np.multiply( normalized_hidden_state, scale_ )
    #print("apply_i_projection: qtz_", qtz_)
    rnd_ = np.round( qtz_ )
    #print("apply_i_projection: rnd_", rnd_)
    # dequantize
    lgpn_ = np.array([[127]])
    lgpn_ = np.broadcast_to( lgpn_, rnd_.shape )
    #print("apply_i_projection: lgpn_", lgpn_)
    min_ = np.minimum( rnd_, lgpn_ )
    lgnn_ = np.array([[-127]])
    lgnn_ = np.broadcast_to( lgnn_, rnd_.shape )
    max_ = np.maximum( min_, lgnn_ )
    #print("apply_i_projection: max_", max_)
    dqtz_ = np.divide( max_, scale_ )
    #print("apply_i_projection: dqtz_", dqtz_)  
    return dqtz_

In [147]:
def weight_quant(w):
    ## Compute the scale factor
    #scale = 1.0 / w.abs().mean().clamp_(min=1e-5)
    ## Quantize and then de-quantize the tensor
    #u = (w * scale).round().clamp_(-1, 1) / scale
    #return u
    abs_ = np.absolute(w)
    #print("weight_quant: mean_=", abs_, abs_.dtype)
    abs_ = abs_.astype(np.float32) # TODO: THIS IS NEEDED!!!
    #print("weight_quant: mean_=", abs_, abs_.dtype)
    mean_ = np.mean(abs_)
    #print("weight_quant: mean_=", mean_, mean_.dtype)
    clamp_ = np.clip(mean_, a_min=1e-5, a_max=None)
    #print("weight_quant: clamp_=", clamp_)
    scale_ = 1.0 / clamp_
    #print("weight_quant: scale=", scale_)
    mult_ = w * scale_   
    #print("weight_quant: mult_-", mult_)
    rnd_ = np.round(mult_)
    #print("weight_quant: rnd=", rnd_)
    clamp_ = np.clip(rnd_,a_min=-1, a_max=1)
    u = clamp_ / scale_
    u = u.astype( w.dtype )
    #print("weight_quant: u=", u, u.shape, u.dtype)
    return u

In [174]:
def apply_i_projection( input_hidden_state, hf_model, layer ):

    # apply rmsnorm before projection - produces 8 bit quantized hidden state
    rmsnorm_weights = hf_model[ "model.layers.%d.attn.i_proj.norm.weight" % layer ]
    q_norm_hidden_state = layer_norm_fwd_quant_kernel( input_hidden_state, rmsnorm_weights )
    q_norm_hidden_state = q_norm_hidden_state.astype( np.float32) # WHY?
    #print("apply_i_projection: q_norm_hidden_state=", q_norm_hidden_state,
    #      q_norm_hidden_state.shape, q_norm_hidden_state.dtype)

    # get ternary quantized projection weights
    linear_weight = hf_model[ "model.layers.%d.attn.i_proj.weight" % layer ]
    #print("apply_i_projection: linear_weight before ternary", linear_weight)
    ternary_linear_weight = weight_quant(linear_weight)
    ternary_linear_weight = ternary_linear_weight.astype(np.float32) # TODO: needed?
    #print("apply_i_projection: ternary_linear_weight", ternary_linear_weight, 
    #      ternary_linear_weight.shape, ternary_linear_weight.dtype)

    # apply projection - TODO: use adds
    i_projected = np.matmul( q_norm_hidden_state, ternary_linear_weight.transpose())
    i_projected = i_projected.astype( input_hidden_state.dtype ) # TODO: needed?
    #print("apply_i_projection: i_projected=", i_projected, i_projected.shape, i_projected.dtype)

    return i_projected

In [175]:
# Applies F projection only

def apply_f_projection( input_hidden_state, hf_model, layer ):

    # apply rmsnorm before projection - produces 8 bit quantized hidden state
    rmsnorm_weights = hf_model[ "model.layers.%d.attn.f_proj.norm.weight" % layer ]
    #print("apply_f_projection: rmsnorm_weights=", rmsnorm_weights,
    #      rmsnorm_weights.shape, rmsnorm_weights.dtype)
    q_norm_hidden_state = layer_norm_fwd_quant_kernel( input_hidden_state, rmsnorm_weights )
    q_norm_hidden_state = q_norm_hidden_state.astype( np.float32) # TODO: needed?
    #print("apply_f_projection: q_norm_hidden_state=", q_norm_hidden_state,
    #      q_norm_hidden_state.shape, q_norm_hidden_state.dtype)

    # get ternary quantized projection weights
    linear_weight = hf_model[ "model.layers.%d.attn.f_proj.weight" % layer ]
    #print("apply_f_projection: linear_weight before ternary", linear_weight)
    ternary_linear_weight = weight_quant(linear_weight)
    ternary_linear_weight = ternary_linear_weight.astype(np.float32) #why?
    #print("apply_f_projection: ternary_linear_weight", ternary_linear_weight, 
    #      ternary_linear_weight.shape, ternary_linear_weight.dtype)

    # apply projection - TODO: use adds
    f_projected = np.matmul( q_norm_hidden_state, ternary_linear_weight.transpose())
    f_projected = f_projected.astype( input_hidden_state.dtype ) # TODO: needed?
    #print("apply_f_projection: f_projected=", f_projected, f_projected.shape, f_projected.dtype)

    return f_projected

In [168]:
# Applies sigmoid

def sigmoid( input_hidden_state ):
    input_hidden_state = input_hidden_state.astype(np.float32) # TODO: needed?
    after_sigmoid = 1.0/(1.0 + np.exp(-input_hidden_state))
    after_sigmoid = after_sigmoid.astype(input_hidden_state.dtype) # TODO: needed?
    return after_sigmoid

In [182]:
# Applieds swiglu

def swiglu( X, Y ):
    #print("swiglu: input X=", X, "Y=", Y)
    X_ = X.astype(np.float32) # TODO: needed?
    Y_ = Y.astype(np.float32) # TODO: needed?
    #sig_ = sigmoid( X )
    #mult_ = np.multiply( sig_, Y)
    #mult_ = mult_.astype( X.dtype )
    mult_ = np.multiply( X_, Y_ )
    denom_ = (1.0 + np.exp(-X_))
    div_ = np.divide( mult_, denom_  )
    div_ = div_.astype( X.dtype )
    return div_

In [190]:
# Applies IF projection

def project_to_IF( input_hidden_state, hf_model, num_heads, layer ):
    i_projected = apply_i_projection(input_hidden_state, hf_model, layer )
    print("project_to_IF: i_projected=", i_projected, i_projected.dtype)
    f_projected = apply_f_projection(input_hidden_state, hf_model, layer )
    print("project_to_IF: f_projected=", f_projected, f_projected.dtype)
    
    f_after_sigmoid = sigmoid( f_projected )
    print("project_to_IF: f_after_sigmoid=", f_after_sigmoid, f_after_sigmoid.dtype)

    i_after_swiglu = swiglu( i_projected, 1 - f_after_sigmoid )  #TODO: slight mismatch with torch
    print("project_to_IF: i_after_swiglu=", i_after_swiglu, i_after_swiglu.dtype)

    # reshape to num_heads
    f_heads_hidden_state = np.reshape( f_after_sigmoid, (f_after_sigmoid.shape[0], num_heads, 
                                        f_after_sigmoid.shape[1], f_after_sigmoid.shape[2]) )
    
    i_heads_hidden_state = np.reshape( i_after_swiglu, (i_after_swiglu.shape[0], num_heads, 
                                        i_after_swiglu.shape[1], i_after_swiglu.shape[2]) )
    return (f_heads_hidden_state, i_heads_hidden_state)

In [14]:
# Applies attention to QKV heads

def apply_attention( q_states, k_states, v_states, hf_model, batch_size, q_len, layer ):
    #print("apply_attention:", q_states.dtype, k_states.dtype, v_states.dtype)
    L = q_states.shape[-2]
    S = k_states.shape[-2]
    #print("apply_attention: L,S", L, S)   
    scale_factor = 1 / math.sqrt(q_states.shape[-1])
    #print("scale_factor=", scale_factor, "via", q_states.shape[-1] )
    attn_bias = np.zeros((L, S), dtype=np.float32 ) #dtype=q_states.dtype) # TODO: why bfloat16 not working?
    #print("apply_attention: attn_bias=", attn_bias.dtype)
    #print("apply_attention: attn bias=", #attn_bias,
    #      attn_bias.shape)
    temp_mask = np.tril( np.ones((L, S), dtype='bool'), k=0 )
    #print("apply_attention: temp mask=", #temp_mask, 
    #      temp_mask.shape)
    #attn_bias.masked_fill_(temp_mask.logical_not(), float("-inf"))
    attn_bias = np.where( temp_mask, attn_bias, np.NINF)
    attn_bias = attn_bias.astype(q_states.dtype)
    #print("apply_attention: attn_bias", #attn_bias, 
    #      attn_bias.shape)
    #print("apply_attention: attn_bias", #attn_bias, 
    #      attn_bias.shape,attn_bias.dtype)
    #attn_weight = query @ key.transpose(-2, -1) * scale_factor
    #print(q_states.shape, k_states.shape, k_states.transpose((0,1,3,2)).shape)
    k_states_transposed = k_states.transpose((0,1,3,2))
    attn_weight_unscaled = np.matmul(q_states, k_states_transposed )
    attn_weight_unscaled = attn_weight_unscaled.astype( q_states.dtype)
    #print("attn_weight unscaled=", #attn_weight_unscaled, 
    #      attn_weight_unscaled.shape, attn_weight_unscaled.dtype)
    attn_weight =  attn_weight_unscaled * scale_factor
    #attn_weight += attn_bias
    attn_weight = np.add( attn_weight, attn_bias)
    attn_weight = attn_weight.astype( q_states.dtype ) # why need to coerce to bfloat16 ?
    #print("apply_attention: attn_weight=", #attn_weight, 
    #      attn_weight.shape)
    #attn_weight = torch.softmax(attn_weight, dim=-1)
    exp_ = np.exp(attn_weight)
    #print("apply_attention: exp=", #exp_,
    #      exp_.shape, exp_.dtype)
    sum_ = np.sum(np.exp(attn_weight), axis=3)
    #print("sum_", #sum_,
    #      sum_.shape)
    sum__ = np.expand_dims(sum_, 3)
    sum___ = np.broadcast_to( sum__, (1, 32, q_len, q_len) )
    #print("sum___", #sum___,
    #      sum___.shape)
    #raise Exception("stop!")
    sm = exp_ / sum___
    #print("apply_attention: softmax=", #sm,
    #      sm.shape, sm.dtype)
    check = np.sum(sm, axis=3)
    #print("apply_attention: check=", #check,
    #      check.shape, check.dtype)
    #HF: attn_weight = torch.dropout(attn_weight, dropout_p, train=True)
    #HF: attn = attn_weight @ value
    z_states = np.matmul( sm, v_states)
    z_states = z_states.astype( q_states.dtype ) # why need to coerce to bfloat16?
    #print("apply_attention: attn 1=", attn,
    #      attn.shape)
    #attn_output = attn_output.transpose(1, 2).contiguous()
    attn = z_states.transpose((0,2,1,3))
    #print("apply_attention: attn 2=", #attn,
    #      attn.shape)  
    #HF: attn_output = attn_output.view(bsz, q_len, -1)
    #print(batch_size, q_len, attn.shape[-1])
    attn = attn.reshape( (batch_size, q_len, attn.shape[-2]*attn.shape[-1] ) )
    #print("apply_attention: attn 3=", #attn,
    #      attn.shape)
    o_weights = hf_model[ TORCH_HF_KEY_MAPPING['layers.%d.attention.wo.weight' % layer] ]
    #print("apply_attention: o_weights=", #o_weights, 
    #      o_weights.shape, o_weights.dtype)  
    o_proj = np.matmul( attn, o_weights.transpose() )
    o_proj = o_proj.astype( q_states.dtype)
    #print("apply_attention: o_proj=", #o_proj, 
    #      o_proj.shape, o_proj.dtype)
    return o_proj, z_states

In [15]:
# Applies MLP layers after attention

def apply_mlp( attn, hf_model, residual, layer ):
    # prepare for fully connected
    hidden_state = np.add( residual, attn )
    #print("apply_mlp: hidden_state after residual add=", hidden_state,
    #      hidden_state.shape, hidden_state.dtype)
    residual = hidden_state
    
    # Apply RMS Norm 
    hidden_state = hidden_state.astype(np.float32)
    #print("hidden_state_32=", #hidden_state, 
    #      hidden_state.dtype)
    variance_epsilon= 1e-05
    rmsnorm = hf_model[ TORCH_HF_KEY_MAPPING['layers.%d.ffn_norm.weight' % layer] ]
    #print("rmsnorm=", rmsnorm, 
    #      rmsnorm.shape)
    variance = np.mean( np.power(hidden_state,2), 2, keepdims=True )
    #print("variance=", variance, 
    #      variance.shape) 
    v_sqrt = np.sqrt( variance + variance_epsilon )
    #print("v_sqrt=", #v_sqrt, 
    #      v_sqrt.shape) 
    v_rsqrt = np.reciprocal( v_sqrt  )
    #print("v_rsqrt=", #v_rsqrt, 
    #      v_rsqrt.shape) 
    hidden_state = np.multiply( hidden_state, v_rsqrt )
    #print("hidden_state_mult=", #hidden_state, 
    #      hidden_state.shape)
    normalized_hidden_state = rmsnorm * hidden_state.astype( ml_dtypes.bfloat16 )
    #print("normalized_hidden_state=", #normalized_hidden_state, 
    #      normalized_hidden_state.shape, normalized_hidden_state.dtype)
    down = hf_model[ TORCH_HF_KEY_MAPPING['layers.%d.feed_forward.w2.weight' % layer] ]
    gate = hf_model[ TORCH_HF_KEY_MAPPING['layers.%d.feed_forward.w1.weight' % layer] ]
    up = hf_model[ TORCH_HF_KEY_MAPPING['layers.%d.feed_forward.w3.weight' % layer] ]
    #print("apply_mlp: mlp mats down=", down, down.shape,
    #      "gate=", gate, gate.shape, "up=", up, up.shape)
    after_gate = np.matmul(normalized_hidden_state, gate.transpose())
    after_gate = after_gate.astype( attn.dtype ) # why need coercion?
    #print("apply_mlp: after gate=",  after_gate,
    #      after_gate.shape, after_gate.dtype)
    after_sigmoid = 1.0/(1.0 + np.exp(-after_gate))
    #print("apply_mlp: after_sigmoid=",  after_sigmoid,
    #      after_sigmoid.shape, after_sigmoid.dtype)
    after_silu = np.multiply( after_gate, after_sigmoid)
    #print("apply_mlp: after_silu=",  after_silu,
    #      after_silu.shape, after_silu.dtype)
    after_up = np.matmul(normalized_hidden_state, up.transpose())
    after_up = after_up.astype( attn.dtype ) # why need coercion?
    #print("apply_mlp: after_up=",  after_up,
    #      after_up.shape, after_up.dtype)
    pre_down = np.multiply(after_silu, after_up)
    pre_down = pre_down.astype( attn.dtype ) # why need coercion?
    #print("apply_mlp: pre_down=",  pre_down,
    #      pre_down.shape, pre_down.dtype)
    after_down = np.matmul(pre_down, down.transpose())
    after_down = after_down.astype( attn.dtype ) # why need coercion?
    #print("apply_mlp: after_down=",  after_down,
    #      after_down.shape, after_down.dtype)
    mlp_out = np.add( residual, after_down )
    return mlp_out

In [16]:
# Applies final layer normalization before lm_head

def final_norm( hidden_state, hf_model ):
    hidden_state = hidden_state.astype(np.float32)
    #print("hidden_state_32=", #hidden_state, 
    #      hidden_state.dtype) 
    variance_epsilon= 1e-05
    rmsnorm = hf_model[ TORCH_HF_KEY_MAPPING['norm.weight'] ]
    #print("rmsnorm=", #rmsnorm, 
    #      rmsnorm.shape)
    variance = np.mean( np.power(hidden_state,2), 2, keepdims=True )
    #print("variance=", #variance, 
    #      variance.shape)
    v_sqrt = np.sqrt( variance + variance_epsilon )
    #print("v_sqrt=", #v_sqrt, 
    #      v_sqrt.shape)
    v_rsqrt = np.reciprocal( v_sqrt  )
    #print("v_rsqrt=", #v_rsqrt, 
    #      v_rsqrt.shape)
    hidden_state = np.multiply( hidden_state, v_rsqrt )
    #print("hidden_state_mult=", #hidden_state, 
    #      hidden_state.shape)
    normalized_hidden_state = rmsnorm * hidden_state.astype( ml_dtypes.bfloat16 )
    #print("apply_rms_norm_pre_attention: normalized_hidden_state=", #normalized_hidden_state, 
    #      normalized_hidden_state.shape,normalized_hidden_state.dtype)
    return normalized_hidden_state

In [17]:
# Applies lm_head/logits processing

def final_prediction(final_hidden_state, hf_model):
    lm_head = hf_model[ TORCH_HF_KEY_MAPPING['output.weight'] ]
    logits = np.matmul(final_hidden_state, lm_head.transpose())
    #print("GW final_logits logits=", logits,
    #    logits.shape, logits.dtype)
    last_logit = logits[-1,-1,:]
    #print("GW final_logits last_logit=", last_logit,
    #    last_logit.shape, last_logit.dtype) 
    # probs = nn.functional.softmax(next_token_scores, dim=-1)
    exp_ = np.exp(last_logit)
    #print("GW final_logits exp=", exp_, exp_.shape, exp_.dtype)
    sum_ = np.sum(np.exp(last_logit), axis=0)
    #print("GW final_logits sum=", sum_, sum_.shape, sum_.dtype)
    sm = exp_ / sum_
    #print("GW final_logits sum=", sm, sm.shape, sm.dtype)
    max_id = np.argmax(sm)
    #print("GW final_logits argmax=", max_id, sm[max_id])
    return max_id

In [18]:
# MAIN: tokenize a prompt

#create_model_key_mappings()
tokenizer = create_tokenizer()
# tokenize prompt
prompt = ["In a shocking finding, scientist discovered a herd of unicorns living in a remote, "]
#prompt = ['''<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n'''
#             '''Cutting Knowledge Date: December 2023\nToday Date: 26 Jul 2024\n\n'''
#             '''You are a pirate chatbot who always responds in pirate speak!'''
#             '''<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n'''
#             '''Who are you?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n''']
token_ids, batch_size, q_len = tokenize_prompt(prompt, tokenizer)

# HACK - why do we need to prepend 1 to match the origin code?
#pre_token = np.array([[1]], dtype=np.uint32)
#print(pre_token.shape, pre_token.dtype, token_ids.shape, token_ids.dtype)
#token_ids = np.append(pre_token, token_ids)
#q_len += 1
print("main: initial token_ids=", token_ids, type(token_ids), q_len)

main: initial token_ids= [[    1   560   264 10382   288  7484 28725 24480  8324   264   559 28715
    302   521   294  1334 28713  3687   297   264  9308 28725 28705]] <class 'numpy.ndarray'> 23


In [19]:
# MAIN: load the model

hf_model = load_hf_export_model()
print(list(hf_model.keys()))

['lm_head.norm.weight', 'lm_head.weight', 'model.embeddings.weight', 'model.layers.0.attn.f_proj.norm.weight', 'model.layers.0.attn.f_proj.weight', 'model.layers.0.attn.g_norm.weight', 'model.layers.0.attn.g_proj.norm.weight', 'model.layers.0.attn.g_proj.weight', 'model.layers.0.attn.i_proj.norm.weight', 'model.layers.0.attn.i_proj.weight', 'model.layers.0.attn.o_proj.norm.weight', 'model.layers.0.attn.o_proj.weight', 'model.layers.0.attn_norm.weight', 'model.layers.0.mlp.down_proj.norm.weight', 'model.layers.0.mlp.down_proj.weight', 'model.layers.0.mlp.gate_proj.norm.weight', 'model.layers.0.mlp.gate_proj.weight', 'model.layers.0.mlp_norm.weight', 'model.layers.1.attn.f_proj.norm.weight', 'model.layers.1.attn.f_proj.weight', 'model.layers.1.attn.g_norm.weight', 'model.layers.1.attn.g_proj.norm.weight', 'model.layers.1.attn.g_proj.weight', 'model.layers.1.attn.i_proj.norm.weight', 'model.layers.1.attn.i_proj.weight', 'model.layers.1.attn.o_proj.norm.weight', 'model.layers.1.attn.o_proj

In [81]:
# Check embedding weights
tensor = hf_model["model.embeddings.weight"]
print("model.embeddings.weight", tensor, tensor.shape, tensor.dtype)

tensor = hf_model["model.layers.0.attn_norm.weight"]
print("model.layers.0.attn_norm.weight",tensor, tensor.shape, tensor.dtype)

tensor = hf_model["model.layers.0.attn.i_proj.norm.weight"]
print("model.layers.0.attn.i_proj.norm.weightt",tensor, tensor.shape, tensor.dtype)

tensor = hf_model["model.layers.0.attn.i_proj.weight"]
print("model.layers.0.attn.i_proj.weight",tensor, tensor.shape, tensor.dtype)

tensor = hf_model["model.layers.0.attn.f_proj.norm.weight"]
print("model.layers.0.attn.f_proj.norm.weight",tensor, tensor.shape, tensor.dtype)

tensor = hf_model["model.layers.0.attn.f_proj.weight"]
print("model.layers.0.attn.f_proj.weight",tensor, tensor.shape, tensor.dtype)

tensor = hf_model["model.layers.0.attn.g_norm.weight"]
print("model.layers.0.attn.g_norm.weight",tensor, tensor.shape, tensor.dtype)

tensor = hf_model["model.layers.0.attn.g_proj.norm.weight"]
print("model.layers.0.attn.g_proj.norm.weight",tensor, tensor.shape, tensor.dtype)




model.embeddings.weight [[-0.0480957 -0.103027 0.0280762 ... -0.0130005 -0.0456543 0.0349121]
 [-0.0742188 -0.0957031 -0.183594 ... -0.222656 -0.0527344 -0.0620117]
 [0.204102 0.239258 0.0336914 ... -0.535156 0.0673828 -0.0132446]
 ...
 [-0.00341797 -0.0373535 -0.0634766 ... -0.00982666 0.116211 0.0493164]
 [0.0727539 -0.192383 -0.0766602 ... 0.172852 -0.212891 -0.0515137]
 [-0.0246582 -0.15332 0.0634766 ... 0.00144196 -0.0522461 0.020874]] (32000, 1024) bfloat16
model.layers.0.attn_norm.weight [0.582031 0.519531 0.566406 ... 0.59375 0.546875 0.570312] (1024,) bfloat16
model.layers.0.attn.i_proj.norm.weightt [0.546875 0.660156 0.582031 ... 0.640625 0.625 0.96875] (1024,) bfloat16
model.layers.0.attn.i_proj.weight [[-0.074707 0.0412598 -0.433594 ... -0.217773 0.216797 0.78125]
 [0.126953 0.102539 0.660156 ... -0.180664 0.738281 0.205078]
 [0.0375977 -0.324219 -0.186523 ... 0.78125 0.296875 0.227539]
 ...
 [0.0673828 0.200195 -0.523438 ... -0.0693359 -0.15918 -0.0849609]
 [0.392578 -0.92

In [21]:
# Check first normalization weights
tensor = hf_model["model.layers.0.attn_norm.weight"]
print(tensor, tensor.shape, tensor.dtype)

#tensor([0.5820, 0.5195, 0.5664,  ..., 0.5938, 0.5469, 0.5703],
#       dtype=torch.float16, requires_grad=True) B= None

[0.582031 0.519531 0.566406 ... 0.59375 0.546875 0.570312] (1024,) bfloat16


In [189]:
# MAIN: Forward all layers with a decode stop criteria

counter=0

#while True:
if True:

    print("Getting next token...", end='')
    
    # create token embeddings
    token_embeddings = token_ids__2__token_embeddings(token_ids, hf_model)
    q_len = token_ids.shape[0]
    
    # initialize first layer input from token_embeddings
    hidden_state = token_embeddings
    
    for layer in range(NUM_LAYERS):

        #pass
        #print("%d-" %layer , end='')
        
        # apply pre-attention layer norm
        residual, normalized_hidden_state = apply_rms_norm_pre_attention( hidden_state, hf_model, layer)
        
        # apply IF projections
        i_heads_hidden_state, f_heads_hidden_state = 
            project_to_IF( normalized_hidden_state, hf_model, NUM_HEADS, layer )
        #q_state, k_state, v_state = project_to_qkv( normalized_hidden_state, hf_model, batch_size, q_len, layer)
        
        # apply attention
        #attn_out, z_state = apply_attention( q_state, k_state, v_state, hf_model, batch_size, q_len, layer)
        
        # apply MLP
        #mlp_out = apply_mlp( attn_out, hf_model, residual, layer)
    
        # prepare for next layer in the loop
        #hidden_state = mlp_out

        break
    
    #normalized_hidden_state = final_norm( hidden_state, hf_model )
    
    #predicted_token_id = final_prediction( normalized_hidden_state, hf_model)
    #print("main: predicted=", predicted_token_id)

    #token_ids = np.append(token_ids, np.array([[predicted_token_id]]) )
    #print("new token_ids=", token_ids)
    #token_ids = token_ids.reshape(1, token_ids.size)
    #print("new token_ids shape=", token_ids.shape)
    #print("main: predicted=", predicted_token_id)

    #new_text = tokenizer.decode(token_ids[0,:])
    #print()
    #print("Done. Here is the new_text-->\n", new_text,"<--")
    #print()

    #counter += 1
    #if counter>50:
    #    break
        
print("Done.")

Getting next token...apply_rms_norm_pre_attention: normalized_hidden_state= [[[-0.333984 -0.384766 -0.804688 ... -1.02344 -0.223633 -0.273438]
  [0.527344 -0.511719 0.636719 ... 0.203125 -0.339844 0.161133]
  [0.419922 -1.10938 0.480469 ... -0.246094 0.785156 0.0991211]
  ...
  [0.237305 -0.472656 0.328125 ... 0.277344 0.367188 0.0795898]
  [0.129883 -0.613281 0.644531 ... -0.0301514 -0.263672 0.613281]
  [1.0625 -0.0366211 0.326172 ... -0.0717773 -0.257812 0.251953]]] (1, 23, 1024) bfloat16
project_to_IF: i_projected= [[[19.125 -4.4375 11.5 ... 6.84375 7.21875 15.375]
  [-0.189453 2.17188 12.625 ... 6.40625 17.375 5.53125]
  [2.5 13.9375 -2 ... 2.4375 10.75 14.4375]
  ...
  [17.75 3.6875 28.25 ... 7.625 16.125 9]
  [7.78125 1.36719 -2.20312 ... 6.71875 9.125 7]
  [5.4375 9.625 -1.64062 ... 18 23.125 9.9375]]] bfloat16
project_to_IF: f_projected= [[[-2.57812 -3.5 -4.03125 ... -6.09375 -5.46875 -5.0625]
  [-6.9375 -3.25 -4.96875 ... -1.35938 -3.65625 -5.3125]
  [-6.5 -5.46875 -2.90625 .